In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path
from sklearn.linear_model import LinearRegression

# paths relative to notebook location
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data" / "cleaned"
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# remove top and right borders from all plots
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [ ]:
# load cleaned merged datasets for each education level
primary = pd.read_csv(DATA_DIR / "primary_gender_merged.csv")
secondary = pd.read_csv(DATA_DIR / "secondary_gender_merged.csv")
tertiary = pd.read_csv(DATA_DIR / "tertiary_gender_merged.csv")

In [ ]:
# quick look at each dataset — shape, column types, and missing values
datasets = {
    "Primary": primary,
    "Secondary": secondary,
    "Tertiary": tertiary
}

for name, df in datasets.items():
    print(f"\n{name} dataset")
    print(df.head())
    print(df.info())
    print(df.isna().sum())

In [ ]:
# columns come in as strings from the CSV, convert to numeric
for df in [primary, secondary, tertiary]:
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["male_enrollment"] = pd.to_numeric(df["male_enrollment"], errors="coerce")
    df["female_enrollment"] = pd.to_numeric(df["female_enrollment"], errors="coerce")

In [ ]:
# gender gap = male minus female (positive means male higher, negative means female higher)
for df in [primary, secondary, tertiary]:
    df["gender_gap"] = df["male_enrollment"] - df["female_enrollment"]

# add level label so we can stack all three datasets later
primary["level"] = "Primary"
secondary["level"] = "Secondary"
tertiary["level"] = "Tertiary"

In [ ]:
# load into in-memory SQLite so we can run SQL queries for EDA
con = sqlite3.connect(":memory:")

primary.to_sql("primary_table", con, index=False, if_exists="replace")
secondary.to_sql("secondary_table", con, index=False, if_exists="replace")
tertiary.to_sql("tertiary_table", con, index=False, if_exists="replace")

In [ ]:
# check how many rows have non-missing enrollment values per level
coverage_primary = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(male_enrollment) AS non_missing_male,
    COUNT(female_enrollment) AS non_missing_female
FROM primary_table
""", con)

coverage_secondary = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(male_enrollment) AS non_missing_male,
    COUNT(female_enrollment) AS non_missing_female
FROM secondary_table
""", con)

coverage_tertiary = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(male_enrollment) AS non_missing_male,
    COUNT(female_enrollment) AS non_missing_female
FROM tertiary_table
""", con)

print("Primary coverage")
display(coverage_primary)

print("Secondary coverage")
display(coverage_secondary)

print("Tertiary coverage")
display(coverage_tertiary)

In [ ]:
# average male enrollment, female enrollment, and gender gap per level
summary_primary = pd.read_sql_query("""
SELECT
    AVG(male_enrollment) AS avg_male,
    AVG(female_enrollment) AS avg_female,
    AVG(gender_gap) AS avg_gap
FROM primary_table
""", con)

summary_secondary = pd.read_sql_query("""
SELECT
    AVG(male_enrollment) AS avg_male,
    AVG(female_enrollment) AS avg_female,
    AVG(gender_gap) AS avg_gap
FROM secondary_table
""", con)

summary_tertiary = pd.read_sql_query("""
SELECT
    AVG(male_enrollment) AS avg_male,
    AVG(female_enrollment) AS avg_female,
    AVG(gender_gap) AS avg_gap
FROM tertiary_table
""", con)

print("Primary summary")
display(summary_primary)

print("Secondary summary")
display(summary_secondary)

print("Tertiary summary")
display(summary_tertiary)

In [ ]:
# tertiary breakdown by region — average gap across all years
# ordered by avg_gap descending so regions with male advantage show first
tertiary_region_summary = pd.read_sql_query("""
SELECT
    country_name,
    ROUND(AVG(male_enrollment), 2) AS avg_male,
    ROUND(AVG(female_enrollment), 2) AS avg_female,
    ROUND(AVG(gender_gap), 2) AS avg_gap
FROM tertiary_table
GROUP BY country_name
ORDER BY avg_gap DESC
""", con)

display(tertiary_region_summary)

In [ ]:
# reusable function to plot male vs female enrollment over time for a given dataset
def plot_gender_trends(df, title, filename):
    yearly = (
        df.groupby("year")[["male_enrollment", "female_enrollment"]]
        .mean()
        .reset_index()
    )

    plt.figure(figsize=(10, 6))
    plt.plot(yearly["year"], yearly["male_enrollment"], label="Male")
    plt.plot(yearly["year"], yearly["female_enrollment"], label="Female")
    plt.title(f"{title} Enrollment Over Time")
    plt.xlabel("Year")
    plt.ylabel("Gross Enrollment (%)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# generate enrollment trend plots for all three levels
plot_gender_trends(primary, "Primary", "primary_gender_trends.png")
plot_gender_trends(secondary, "Secondary", "secondary_gender_trends.png")
plot_gender_trends(tertiary, "Tertiary", "tertiary_gender_trends.png")

In [ ]:
# reusable function to plot the gender gap over time for a given dataset
# dashed line at 0 shows the parity threshold
def plot_gender_gap(df, title, filename):
    gap = (
        df.groupby("year")["gender_gap"]
        .mean()
        .reset_index()
    )

    plt.figure(figsize=(10, 6))
    plt.plot(gap["year"], gap["gender_gap"])
    plt.axhline(0, linestyle="--")
    plt.title(f"{title} Gender Gap Over Time (Male - Female)")
    plt.xlabel("Year")
    plt.ylabel("Gender Gap")
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# generate gender gap plots for all three levels
plot_gender_gap(primary, "Primary", "primary_gap_trend.png")
plot_gender_gap(secondary, "Secondary", "secondary_gap_trend.png")
plot_gender_gap(tertiary, "Tertiary", "tertiary_gap_trend.png")

In [ ]:
# stack all three levels into one dataframe for cross-level comparisons
combined = pd.concat([primary, secondary, tertiary], ignore_index=True)

In [ ]:
# female enrollment trends by education level on one plot
female_level = (
    combined.groupby(["year", "level"])["female_enrollment"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(10, 6))

for level in female_level["level"].unique():
    subset = female_level[female_level["level"] == level]
    plt.plot(subset["year"], subset["female_enrollment"], label=level)

plt.title("Female Enrollment by Education Level")
plt.xlabel("Year")
plt.ylabel("Gross Enrollment (%)")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "female_enrollment_by_level.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# same plot but for male enrollment
male_level = (
    combined.groupby(["year", "level"])["male_enrollment"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(10, 6))

for level in male_level["level"].unique():
    subset = male_level[male_level["level"] == level]
    plt.plot(subset["year"], subset["male_enrollment"], label=level)

plt.title("Male Enrollment by Education Level")
plt.xlabel("Year")
plt.ylabel("Gross Enrollment (%)")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "male_enrollment_by_level.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# gender gap across all three levels on one plot
# shows how the gap has evolved differently at each level
gap_level = (
    combined.groupby(["year", "level"])["gender_gap"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(10, 6))

for level in gap_level["level"].unique():
    subset = gap_level[gap_level["level"] == level]
    plt.plot(subset["year"], subset["gender_gap"], label=level)

plt.axhline(0, linestyle="--")
plt.title("Gender Gap by Education Level")
plt.xlabel("Year")
plt.ylabel("Male - Female Enrollment")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "gender_gap_by_level.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# filter to the most recent year for regional snapshots
latest_year = combined["year"].max()
latest = combined[combined["year"] == latest_year].copy()

In [ ]:
# bar chart of female tertiary enrollment by region for the latest year
tertiary_latest_female = (
    latest[latest["level"] == "Tertiary"]
    .sort_values("female_enrollment")
)

plt.figure(figsize=(10, 7))
plt.barh(tertiary_latest_female["country_name"], tertiary_latest_female["female_enrollment"])
plt.title(f"Female Tertiary Enrollment by Region ({int(latest_year)})")
plt.xlabel("Gross Enrollment (%)")
plt.tight_layout()
plt.savefig(FIG_DIR / "female_tertiary_latest_region.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# bar chart of tertiary gender gap by region for the latest year
# negative bars = female outenrolling male
tertiary_latest_gap = (
    latest[latest["level"] == "Tertiary"]
    .sort_values("gender_gap")
)

plt.figure(figsize=(10, 7))
plt.barh(tertiary_latest_gap["country_name"], tertiary_latest_gap["gender_gap"])
plt.axvline(0, linestyle="--")
plt.title(f"Tertiary Gender Gap by Region ({int(latest_year)})")
plt.xlabel("Male - Female Enrollment")
plt.tight_layout()
plt.savefig(FIG_DIR / "tertiary_gap_latest_region.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
# fit a simple linear regression of enrollment on year
# slope tells us the average annual change, R^2 tells us how linear the trend is
def run_trend_model(df, outcome, label):
    model_df = df.dropna(subset=["year", outcome]).copy()

    yearly = (
        model_df.groupby("year")[outcome]
        .mean()
        .reset_index()
    )

    X = yearly[["year"]]
    y = yearly[outcome]

    model = LinearRegression()
    model.fit(X, y)

    print(label)
    print("Slope:", round(model.coef_[0], 4))
    print("Intercept:", round(model.intercept_, 4))
    print("R^2:", round(model.score(X, y), 4))
    print()

In [ ]:
# female enrollment trends across all three levels
run_trend_model(primary, "female_enrollment", "Primary female enrollment trend")
run_trend_model(secondary, "female_enrollment", "Secondary female enrollment trend")
run_trend_model(tertiary, "female_enrollment", "Tertiary female enrollment trend")

In [ ]:
# male enrollment trends across all three levels
run_trend_model(primary, "male_enrollment", "Primary male enrollment trend")
run_trend_model(secondary, "male_enrollment", "Secondary male enrollment trend")
run_trend_model(tertiary, "male_enrollment", "Tertiary male enrollment trend")